# 11 — Micro/Macro ERSP Comparison (MicroEPI patients)

For each MicroEPI patient, compares ERSPs between micro-electrode tetrodes and the first macro contact of the same shaft.

**Inputs**
- `.mat` export(s) with `dataEcog` (macros), `dataMicroDown` (micros @ 2048 Hz), `photodiode`, `chansEcog`, `chansMicro`
- behavioral TSV in the same folder as the `.mat`(s)
- BIDS electrodes TSV for WM channel derivation

**Outputs** → `01_FBM_Analysis/outputs/11_MicroMacro_results/<patient_id>/`
- `prep0/` — per-condition timing TSVs (written by the pipeline)
- per-shaft macro ERSP figure
- per-tetrode 2×2 micro ERSP figures

**Reused**: `lf_trials`, `lf_ersp`, `LFfunctions_PDextract`. **New**: `lf_micromacro`.

**Notes**: Notch filtering for 50 Hz harmonics is deferred (TODO). ERSP `fmax` is raised to 1000 Hz locally for this notebook only.

## 1. Imports

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd

# Make ../functions and the uploads dir (containing LFfunctions_PDextract) importable
sys.path.insert(0, str(Path('../functions').resolve()))

import lf_micromacro as mm
from lf_trials import collect_trials
from lf_ersp import ERSPParams, compute_ersp
from LFfunctions_PDextract import _read_trial_table, save_onsets_offsets_by_condition

## 2. Per-patient configuration

One entry per patient. All paths use the `nasac-m2` server prefix.

In [ ]:
FS_ECOG = 2048  # Hz — dataEcog rate; dataMicroDown already matches

SERVER_ROOT = r"\\nasac-m2.unige.ch\m-HumanNeuronLab"
BIDS_ELEC_ROOT = SERVER_ROOT + r"\DATARAW\BIDS_elec\MICROEPI"

PATIENTS = {
    "MicroEPI-G-01": {
        "data_dir":       SERVER_ROOT + r"\DATARAW\MICROEPI\MicroEPI-G-01\task_otherlabs\exp_Lora1FLM_all",
        "mat_files":      ["f0001_export_Labs_phmicrodown.mat",
                           "f0002_export_Labs_phmicrodown.mat"],
        "tsv_file":       "sub-MicroEpiG01_task-LanguageMapping_timestamp-28-11-2024(15h52m8s)_lang-FRE_events.tsv",
        "electrodes_tsv": BIDS_ELEC_ROOT + r"\sub-5515\ieeg\*_electrodes.tsv",
    },
    "MicroEPI-G-02": {
        "data_dir":       SERVER_ROOT + r"\DATARAW\MICROEPI\MicroEPI-G-02\task_otherlabs\Lora_FLM",
        "mat_files":      ["f0001_export_Labs_ph.mat",
                           "f0002_export_Labs_ph.mat"],
        "tsv_file":       "MicroEpi-G-02_sync_micromedBlackrock_20250416_onsets_offsets_FLM_v2.tsv",
        "electrodes_tsv": BIDS_ELEC_ROOT + r"\sub-5533\ieeg\*_electrodes.tsv",
    },
    "MicroEPI-G-03": {
        "data_dir":       SERVER_ROOT + r"\DATARAW\MICROEPI\MicroEPI-G-03\task_FBM\exp3_Lora1_LM_CLA_CLV",
        "mat_files":      ["f0001_export_Labs_ph.mat",
                           "f0002_export_Labs_ph.mat",
                           "f0003_export_Labs_ph.mat"],
        "tsv_file":       "sub-microepig03_task-LanguageMapping_timestamp-7-3-2025(14h59m34s)_lang-FRE_events.tsv",
        "electrodes_tsv": BIDS_ELEC_ROOT + r"\sub-6619\ieeg\*_electrodes.tsv",
    },
    "MicroEPI-G-04": {
        "data_dir":       SERVER_ROOT + r"\DATARAW\MICROEPI\MicroEPI-G-04\task_FBM\data_LM\exp4_Lora",
        "mat_files":      ["f0001_export_Labs_ph.mat",
                           "f0002_export_Labs_ph.mat"],
        "tsv_file":       "sub-MicroEpi-G-04_task-LanguageMapping_timestamp-7-4-2025(14h55m53s)_lang-ENG_events.tsv",
        "electrodes_tsv": BIDS_ELEC_ROOT + r"\sub-6704\ieeg\*_electrodes.tsv",
    },
    "MicroEPI-G-05": {
        "data_dir":       SERVER_ROOT + r"\DATARAW\MICROEPI\MicroEPI-G-05\tasks\exp9_JonathanFLM_2025_06_18",
        "mat_files":      ["f0001_export_Labs_phmicrodown.mat",
                           "f0002_export_Labs_phmicrodown.mat"],
        "tsv_file":       "sub-microepi-g-05_task-LanguageMapping_datetime-18-6-2025(15h39m19s)_language-FRE_events.tsv",
        "electrodes_tsv": BIDS_ELEC_ROOT + r"\sub-6684\ieeg\*_electrodes.tsv",
    },
    "MicroEPI-G-06": {
        "data_dir":       SERVER_ROOT + r"\DATARAW\MICROEPI\MicroEPI-G-06\tasks\exp1_lora_2026_01_29_withmicro",
        "mat_files":      ["f0001_export_Labs_phmicrodown.mat",
                           "f0002_export_Labs_phmicrodown.mat"],
        "tsv_file":       "sub-6854_task-LanguageMapping_datetime-29-1-2026(17h33m57s)_language-FRE_events.tsv",
        "electrodes_tsv": BIDS_ELEC_ROOT + r"\sub-6854\ieeg\*_electrodes.tsv",
    },
}

OUTPUT_ROOT = Path("../outputs/11_MicroMacro_results")

# ERSP parameters — fmax raised to 1000 Hz for this notebook only
ERSP_PARAMS = ERSPParams(fmax=1000.0)

# Toggles
APPLY_WM_TO_MICROS = False   # default: WM reref only on macros
CONDITIONS         = ["picture", "audio", "reading"]
ERSP_MODE          = "TN"   # "TN" (warped) or "RT"

## 3. Per-patient pipeline

For each patient:
1. Load + concatenate `.mat` files.
2. Extract photodiode on/off events (square-wave detection from `LFfunctions_PDextract`).
3. Read behavioral TSV, align first trial to first PD onset, save per-condition timing TSVs to `prep0/`.
4. `collect_trials` reads those TSVs back (normalized, filtered, QC'd).
5. Build combined (macro+micro) signal matrix; WM reref (macros-only by default).
6. Pair shafts; group micros into tetrodes.
7. For every condition × shaft × tetrode: compute ERSPs; save 1 macro figure + 2×2 tetrode figures.

In [ ]:
def process_patient(pid, cfg, *, ersp_mode=ERSP_MODE, params=ERSP_PARAMS,
                    apply_wm_to_micros=APPLY_WM_TO_MICROS):
    print(f"\n=== {pid} ===")
    out_dir  = OUTPUT_ROOT / pid
    prep_dir = out_dir / "prep0"
    prep_dir.mkdir(parents=True, exist_ok=True)

    # 1) Load + concatenate
    d = mm.load_and_concatenate_mats(cfg["data_dir"], cfg["mat_files"])
    fs = d["fs"]
    print(f"  signals: ecog {d['data_ecog'].shape} | micro {d['data_micro'].shape} | fs={fs}")

    # 2) Photodiode events
    on_abs, off_abs = mm.extract_events_from_photodiode(d["photodiode"], fs)
    print(f"  photodiode: {len(on_abs)} onsets, {len(off_abs)} offsets")

    # 3) Behavioral TSV -> save per-condition timing TSVs
    beh_path = os.path.join(cfg["data_dir"], cfg["tsv_file"])
    beh = _read_trial_table(beh_path)
    dfl = beh["raw_df"].rename(columns=str.lower)

    def _pick(cols):
        return next((dfl[c].astype(str).to_numpy() for c in cols if c in dfl), None)

    condition_name = _pick(["category", "blockname"])
    resp_accuracy  = _pick(["response_type", "responseaccuracy"])
    trial_idx_col  = _pick(["exemplar", "stimnumber"])

    short = {"picture": "pict", "auditory": "audi", "reading": "read"}
    trial_ids = [short.get(str(x).lower().split("_")[0], str(x).lower())
                 for x in (condition_name if condition_name is not None else [])]

    save_onsets_offsets_by_condition(
        patient_id=pid, block_name="LM",
        onsets=on_abs, offsets=off_abs, sampling_rate=fs,
        trial_ids=trial_ids, out_dir=str(prep_dir),
        condition_name=condition_name, resp_accuracy=resp_accuracy, trial_idx=trial_idx_col,
    )

    # 4) Read back with validation / outlier trimming
    cond_groups = collect_trials(
        str(prep_dir), fs_hz=fs, patient_id=pid,
        report_path=str(prep_dir / f"{pid}_trialQC.tsv"),
        outlier_method="IQR", iqr_k=1.5,
    )
    print(f"  conditions found: {list(cond_groups.keys())}")

    # 5) Combined signals + WM reref
    signals, names, is_micro = mm.build_combined_signals(
        d["data_ecog"], d["data_micro"], d["chans_ecog"], d["chans_micro"])
    wm_names = mm.derive_wm_channels_from_electrodes_tsv(cfg["electrodes_tsv"])
    print(f"  WM channels derived: {len(wm_names)} -> {wm_names[:6]}{'...' if len(wm_names)>6 else ''}")
    signals, wm_used, wm_excl = mm.apply_wm_reref_selective(
        signals, names, wm_names, is_micro, apply_wm_to_micros=apply_wm_to_micros)
    print(f"  WM reref applied: used={len(wm_used)}, excluded={len(wm_excl)}, micros={'yes' if apply_wm_to_micros else 'no'}")

    # TODO: adaptive 50 Hz harmonic notch for micros (up to 1000 Hz) — deferred

    # 6) Pair shafts and group tetrodes
    shafts = mm.pair_micro_to_macro(d["chans_micro"], d["chans_ecog"])
    print(f"  shafts paired: {len(shafts)} -> {list(shafts.keys())}")

    # 7) ERSPs per condition x shaft x tetrode
    name_to_idx = {nm: i for i, nm in enumerate(names)}
    for cond, (on, off, tend) in cond_groups.items():
        if cond not in CONDITIONS or len(on) == 0:
            continue
        cond_dir = out_dir / cond
        cond_dir.mkdir(parents=True, exist_ok=True)

        for shaft, pair in shafts.items():
            # --- macro ERSP
            macro_idx = name_to_idx[pair["macro"]]
            macro_ersp = compute_ersp(
                signals, fs, on, off, macro_idx,
                trial_ends=tend, mode=ersp_mode, params=params)
            mm.plot_macro_ersp(
                macro_ersp,
                save_path=str(cond_dir / f"{shaft}_macro_{pair['macro']}.tif"),
                patient_id=pid, condition=cond, chan_name=pair["macro"], params=params)

            # --- tetrodes
            tetrodes = mm.group_into_tetrodes(pair["micros"])
            for t_idx, tet in enumerate(tetrodes, start=1):
                ersps = [compute_ersp(signals, fs, on, off, name_to_idx[nm],
                                      trial_ends=tend, mode=ersp_mode, params=params)
                         for nm in tet]
                mm.plot_tetrode_ersp_2x2(
                    ersps,
                    save_path=str(cond_dir / f"{shaft}_tetrode{t_idx}.tif"),
                    patient_id=pid, condition=cond,
                    tetrode_names=tet, tetrode_idx=t_idx, params=params)

        print(f"  [{cond}] done: {len(shafts)} shafts")

    return out_dir

## 4. Run the pipeline

Run all patients, or a single one by slicing `PATIENTS`.

In [ ]:
# Run all patients
for pid, cfg in PATIENTS.items():
    try:
        process_patient(pid, cfg)
    except Exception as e:
        print(f"[ERROR] {pid}: {type(e).__name__}: {e}")

## 5. Single-patient debug

Uncomment to process just one patient.

In [ ]:
# pid = "MicroEPI-G-06"
# process_patient(pid, PATIENTS[pid])